In [ ]:
!pip install -q \
  torch \
  peft \
  huggingface_hub \
  ipywidgets \
  optuna

!pip install -U datasets

In [2]:
import os, shutil, gc, torch, optuna
from huggingface_hub import login, notebook_login, HfFolder, HfApi, hf_hub_download, delete_repo, list_repo_files, snapshot_download
os.environ["TRANSFORMERS_NO_TF"] = "1"   # prevents TF/Keras import
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig,
                          Trainer, TrainingArguments, DataCollatorWithPadding, default_data_collator)
from peft import (LoraConfig, get_peft_model, prepare_model_for_kbit_training)
from datasets import load_dataset, concatenate_datasets, Dataset, DatasetDict
import json
import inspect
import pandas as pd
import numpy as np
import tempfile
from datetime import datetime
from collections import Counter
from torch.utils.data import Dataset, Subset

ModuleNotFoundError: No module named 'optuna'

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
api = HfApi()
login()

In [ ]:
HF_DATASET_REPO = "eduhuemar001/news"

# Download JSON files from HF dataset repo
train_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename="train.json",
    repo_type="dataset",
    local_dir=".",
    local_dir_use_symlinks=False
)
eval_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename="eval.json",
    repo_type="dataset",
    local_dir=".",
    local_dir_use_symlinks=False
)

# Load datasets
with open(train_path, "r", encoding="utf-8") as f:
    dataset_train = json.load(f)

with open(eval_path, "r", encoding="utf-8") as f:
    dataset_eval = json.load(f)

C:\Users\marku\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:982: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


In [ ]:
class NewsPairDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data                  # list of dicts with keys of title, text, status
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        item = self.data[i]
        title = (item.get("title") or "").strip()
        text  = (item.get("text")  or "").strip()
        label = int(item.get("status"))   # 0/1

        enc = self.tok(
            text=title,                   # News title
            text_pair=text,               # News article
            truncation="only_second",     # keep full title, truncate only news article
            max_length=self.max_length,
            padding=False,                # let collator pad
            return_attention_mask=True
        )

        out = {
            "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(label, dtype=torch.long),
        }
        return out

train_dataset = NewsPairDataset(dataset_train, tokenizer, max_length=512)
eval_dataset  = NewsPairDataset(dataset_eval,  tokenizer, max_length=512)

In [ ]:
rns = np.random.RandomState(42)
TRAIN_CAP = min(len(train_dataset), 1000)
EVAL_CAP  = min(len(eval_dataset),  200)

train_idx = rns.choice(len(train_dataset), size=TRAIN_CAP, replace=False)
eval_idx  = rns.choice(len(eval_dataset),  size=EVAL_CAP,  replace=False)

train_small = Subset(train_dataset, sorted(train_idx))
eval_small  = Subset(eval_dataset,  sorted(eval_idx))

def objective(trial: optuna.Trial):
    # Hyperparameter search space
    hp = {
        "learning_rate":       trial.suggest_float("learning_rate", 1e-5, 5e-3, log=True),
        "weight_decay":        trial.suggest_float("weight_decay", 0.0, 0.2),
        "warmup_ratio":        trial.suggest_float("warmup_ratio", 0.0, 0.2),
        "lr_scheduler_type":   trial.suggest_categorical("lr_scheduler_type", ["linear", "cosine"]),
        "per_device_train_bs": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32]),
        "per_device_eval_bs":  trial.suggest_categorical("per_device_eval_batch_size",  [16, 32]),
        "gradient_accum":      trial.suggest_categorical("gradient_accumulation_steps", [1, 2, 4]),
        "num_train_epochs":    trial.suggest_int("num_train_epochs", 2, 6),
    }

    # ----- model -----
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model = model.to("cuda")
    # print(next(model.parameters()).device)

    args = TrainingArguments(
        output_dir=f"./runs/trial_{trial.number}",
        learning_rate=hp["learning_rate"],
        weight_decay=hp["weight_decay"],
        warmup_ratio=hp["warmup_ratio"],
        lr_scheduler_type=hp["lr_scheduler_type"],
        per_device_train_batch_size=hp["per_device_train_bs"],
        per_device_eval_batch_size=hp["per_device_eval_bs"],
        gradient_accumulation_steps=hp["gradient_accum"],
        num_train_epochs=hp["num_train_epochs"],
        evaluation_strategy="steps",
        eval_steps=200,
        logging_strategy="steps",
        logging_steps=50,
        save_strategy="no",
        report_to="none",
        fp16=False,
        bf16=False,
        seed=42,
        load_best_model_at_end=False,
        dataloader_num_workers=0,
        optim="adamw_torch",
    )

    trainer = Trainer(
        model=model,
        args=args,
        tokenizer=tokenizer,
        train_dataset=train_small,
        eval_dataset=eval_small,
        data_collator=DataCollatorWithPadding(tokenizer),
    )

    trainer.train()
    metrics = trainer.evaluate()
    score = float(metrics["eval_loss"])

    # cleanup VRAM
    del trainer, model
    torch.cuda.empty_cache()

    return score

In [ ]:
HF_REPO = "eduhuemar001/distilbert-news-tuning"
REMOTE_DB_FILE = "optuna_study.db"
LOCAL_DB = f"/content/{REMOTE_DB_FILE}"
STUDY_KEY = f"sqlite:///{LOCAL_DB}"

api = HfApi()

# 1) Try to resume from HF Hub
try:
    db_path = hf_hub_download(
        repo_id=HF_REPO,
        filename=REMOTE_DB_FILE,
        repo_type="model",
        local_dir="/content",
        local_dir_use_symlinks=False
    )
    print(f"Resuming from HF DB at: {db_path}")
except Exception as e:
    print(f"No remote DB found (starting fresh): {e}")
    # ensure local path exists; Optuna will create the DB file on first write
    if not os.path.exists(LOCAL_DB):
        open(LOCAL_DB, "wb").close()

# 2) Create (or resume) persistent study using the local SQLite file
study = optuna.create_study(
    study_name="distilbert_hpo",
    direction="minimize",              # or "maximize" if you optimize F1
    storage=STUDY_KEY,
    load_if_exists=True
)

# 3) Callback: upload updated DB to HF after each trial
def upload_db_after_trial(study, trial):
    # make sure the repo exists
    api.create_repo(repo_id=HF_REPO, repo_type="dataset", exist_ok=True)
    api.upload_file(
        path_or_fileobj=LOCAL_DB,
        path_in_repo=REMOTE_DB_FILE,
        repo_id=HF_REPO,
        repo_type="dataset",
        commit_message=f"Optuna DB update after trial {trial.number}"
    )
    print(f"Pushed DB after trial {trial.number}")

# 4) Run optimization (writes to SQLite every trial; then uploads)
study.optimize(objective, n_trials=20, callbacks=[upload_db_after_trial])

print("Best params:", study.best_trial.params)

[I 2025-11-09 17:03:09,310] A new study created in RDB with name: distilbert_hpo
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\marku\anaconda3\Lib\site-packages\transformers\training_args.py:1494: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Step,Training Loss,Validation Loss


[I 2025-11-09 17:07:02,260] Trial 0 finished with value: 0.0002460479154251516 and parameters: {'learning_rate': 0.00014373196785788194, 'weight_decay': 0.10183347532077046, 'warmup_ratio': 0.04978992780535996, 'lr_scheduler_type': 'cosine', 'per_device_train_batch_size': 8, 'per_device_eval_batch_size': 32, 'gradient_accumulation_steps': 2, 'num_train_epochs': 3}. Best is trial 0 with value: 0.0002460479154251516.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
